# Merge & Decode Facebook Messages
Reads all `message_*.json` files from `Data/PrivateData/`, merges them, sorts by `timestamp_ms`, fixes the Facebook Latin-1-escaped UTF-8 encoding, and writes the result to `Data/EncodedData/messages.json`.

In [8]:
import json
import glob
import os
import base64
import hashlib
import getpass
from pathlib import Path
from cryptography.fernet import Fernet

In [9]:
def fix_encoding(obj):
    """Recursively fix Facebook's mojibake: latin-1 bytes re-encoded as UTF-8."""
    if isinstance(obj, str):
        try:
            return obj.encode('latin-1').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            return obj
    if isinstance(obj, list):
        return [fix_encoding(item) for item in obj]
    if isinstance(obj, dict):
        return {fix_encoding(k): fix_encoding(v) for k, v in obj.items()}
    return obj

In [10]:
base_dir = Path(os.getcwd()).parent  # messagesAnalysis/
input_dir = base_dir / 'Data' / 'PrivateData'
output_dir = base_dir / 'Data' / 'EncodedData'
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Input : {input_dir}")
print(f"Output: {output_dir}")

Input : /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/PrivateData
Output: /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/EncodedData


In [11]:
input_files = sorted(input_dir.glob('message_*.json'))
print(f"Found {len(input_files)} file(s): {[f.name for f in input_files]}")

all_messages = []
participants_set = {}

for path in input_files:
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    data = fix_encoding(data)

    for p in data.get('participants', []):
        participants_set[p['name']] = p

    all_messages.extend(data.get('messages', []))

print(f"Total messages before dedup: {len(all_messages)}")

Found 5 file(s): ['message_1.json', 'message_2.json', 'message_3.json', 'message_4.json', 'message_5.json']
Total messages before dedup: 44422


In [12]:
# Sort ascending by timestamp_ms (oldest first)
all_messages.sort(key=lambda m: m.get('timestamp_ms', 0))

merged = {
    'participants': list(participants_set.values()),
    'messages': all_messages,
}

print(f"Participants : {len(merged['participants'])}")
print(f"Total messages: {len(merged['messages'])}")
print(f"Oldest : {merged['messages'][0].get('timestamp_ms')}")
print(f"Newest : {merged['messages'][-1].get('timestamp_ms')}")

Participants : 37
Total messages: 44422
Oldest : 1768955883993
Newest : 1780454758698


In [13]:
output_path = output_dir / 'messages.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

print(f"Written to {output_path}  ({output_path.stat().st_size / 1024:.1f} KB)")

Written to /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/EncodedData/messages.json  (16140.7 KB)


In [14]:
# Sanity-check: print first 3 messages
for msg in merged['messages'][:3]:
    print(msg)

{'sender_name': 'Bế Minh Nhật', 'timestamp_ms': 1768955883993, 'content': 'Chào cả nhà ^^ mình lập group chat cho buổi Tea & Talk ngày Thứ Bảy 24/1 tới này nha!', 'is_geoblocked_for_viewer': False, 'is_unsent_image_by_messenger_kid_parent': False}
{'sender_name': 'Bế Minh Nhật', 'timestamp_ms': 1768955890861, 'content': 'Yoroshiku cả nhà ạ', 'reactions': [{'reaction': '❤', 'actor': 'Linh Tran Hoang'}], 'is_geoblocked_for_viewer': False, 'is_unsent_image_by_messenger_kid_parent': False}
{'sender_name': 'Bế Minh Nhật', 'timestamp_ms': 1768955908508, 'content': 'Nhật đã đặt tên nhóm là MPKEN | "Lịch sử & Triết học".', 'is_geoblocked_for_viewer': False, 'is_unsent_image_by_messenger_kid_parent': False}


---
## Level 1 — Hash sender names
Password-derived Fernet key. Every `sender_name` / `actor` / participant `name` is replaced with its ciphertext. A lookup table mapping original name → token is stored alongside the data so Level 2 can wrap it too.

In [15]:
def password_to_fernet_key(password: str, salt: bytes = b'philosophy_group_salt_l1') -> Fernet:
    """Derive a Fernet key from a password using PBKDF2."""
    key_bytes = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, iterations=200_000)
    return Fernet(base64.urlsafe_b64encode(key_bytes))

pwd1 = getpass.getpass('Level-1 password (sender names): ')
fernet1 = password_to_fernet_key(pwd1)
print('Level-1 key derived.')

Level-1 key derived.


In [ ]:
# Build a lookup: original name -> encrypted token (deterministic per key)
# We encrypt each name once and reuse the same token everywhere.
all_names = {p['name'] for p in merged['participants']}
name_to_token = {
    name: fernet1.encrypt(name.encode()).decode()
    for name in all_names
}

def replace_names(obj, mapping):
    """Replace every name string in the data with its encrypted token."""
    if isinstance(obj, str):
        return mapping.get(obj, obj)
    if isinstance(obj, list):
        return [replace_names(item, mapping) for item in obj]
    if isinstance(obj, dict):
        return {k: replace_names(v, mapping) for k, v in obj.items()}
    return obj

merged_l1 = replace_names(merged, name_to_token)

# Attach the lookup table so we can decrypt names later
merged_l1['_name_tokens'] = name_to_token

print(f"Replaced {len(name_to_token)} unique sender names with encrypted tokens.")

Replaced 37 unique sender names with encrypted tokens.
Sample: [('Đinh Thị Nghĩa', 'gAAAAABqIXN0q8QvJ7KH0OvqgDQWbV5KEYjPEahA62G-i2JlOZz4glPwAFm_hnVYklD1X6BgtjqN3Unl7-Y0IvDu1d9wri4Xok-1OvNNRpQQCEVz1gIlvVk='), ('Lan Hương', 'gAAAAABqIXN0RnyGYxsJdeY9G7tDWXlZHJ_jrVG5atGH8d2mwA07-Dcba1mWwfW4hpN5CCquteMCtydf6-GWAmerpX3rxIy1Uw==')]


In [17]:
l1_path = output_dir / 'messages_l1.json'
with open(l1_path, 'w', encoding='utf-8') as f:
    json.dump(merged_l1, f, ensure_ascii=False, indent=2)

print(f"Level-1 file written: {l1_path}  ({l1_path.stat().st_size / 1024:.1f} KB)")

Level-1 file written: /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/EncodedData/messages_l1.json  (23927.2 KB)


---
## Level 2 — Encrypt the entire JSON file
The whole Level-1 JSON bytes are encrypted with a second password-derived Fernet key and stored as a single `.enc` file.

In [18]:
def password_to_fernet_key_l2(password: str, salt: bytes = b'philosophy_group_salt_l2') -> Fernet:
    key_bytes = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, iterations=200_000)
    return Fernet(base64.urlsafe_b64encode(key_bytes))

pwd2 = getpass.getpass('Level-2 password (whole file): ')
fernet2 = password_to_fernet_key_l2(pwd2)
print('Level-2 key derived.')

Level-2 key derived.


In [19]:
l1_bytes = json.dumps(merged_l1, ensure_ascii=False).encode('utf-8')
encrypted_blob = fernet2.encrypt(l1_bytes)

l2_path = output_dir / 'messages_l2.enc'
l2_path.write_bytes(encrypted_blob)

print(f"Level-2 file written: {l2_path}  ({l2_path.stat().st_size / 1024:.1f} KB)")

Level-2 file written: /Users/vudang/Documents/Projects/PhilosophyGroup/messagesAnalysis/Data/EncodedData/messages_l2.enc  (27142.1 KB)


---
## Verification — Decrypt both layers and check first 3 messages

In [20]:
# --- Decrypt Layer 2: file -> JSON with hashed names ---
v_pwd2 = getpass.getpass('Verify — Level-2 password: ')
v_fernet2 = password_to_fernet_key_l2(v_pwd2)

raw_bytes = l2_path.read_bytes()
decrypted_l1_bytes = v_fernet2.decrypt(raw_bytes)
decrypted_l1 = json.loads(decrypted_l1_bytes.decode('utf-8'))

print('Level-2 decryption OK.')
print(f"  Messages : {len(decrypted_l1['messages'])}")
print(f"  Participants: {len(decrypted_l1['participants'])}")

Level-2 decryption OK.
  Messages : 44422
  Participants: 37


In [21]:
# --- Decrypt Layer 1: recover original sender names ---
v_pwd1 = getpass.getpass('Verify — Level-1 password: ')
v_fernet1 = password_to_fernet_key(v_pwd1)

token_to_name = {
    token: v_fernet1.decrypt(token.encode()).decode()
    for token in decrypted_l1['_name_tokens'].values()
}

def restore_names(obj, mapping):
    if isinstance(obj, str):
        return mapping.get(obj, obj)
    if isinstance(obj, list):
        return [restore_names(item, mapping) for item in obj]
    if isinstance(obj, dict):
        return {k: restore_names(v, mapping) for k, v in obj.items()}
    return obj

restored = restore_names(decrypted_l1, token_to_name)
print('Level-1 decryption OK.')

Level-1 decryption OK.


In [22]:
# --- Show first 3 messages with original names restored ---
print("First 3 messages after full decryption:\n")
for msg in restored['messages'][:3]:
    print(json.dumps(msg, ensure_ascii=False, indent=2))
    print()

First 3 messages after full decryption:

{
  "sender_name": "Bế Minh Nhật",
  "timestamp_ms": 1768955883993,
  "content": "Chào cả nhà ^^ mình lập group chat cho buổi Tea & Talk ngày Thứ Bảy 24/1 tới này nha!",
  "is_geoblocked_for_viewer": false,
  "is_unsent_image_by_messenger_kid_parent": false
}

{
  "sender_name": "Bế Minh Nhật",
  "timestamp_ms": 1768955890861,
  "content": "Yoroshiku cả nhà ạ",
  "reactions": [
    {
      "reaction": "❤",
      "actor": "Linh Tran Hoang"
    }
  ],
  "is_geoblocked_for_viewer": false,
  "is_unsent_image_by_messenger_kid_parent": false
}

{
  "sender_name": "Bế Minh Nhật",
  "timestamp_ms": 1768955908508,
  "content": "Nhật đã đặt tên nhóm là MPKEN | \"Lịch sử & Triết học\".",
  "is_geoblocked_for_viewer": false,
  "is_unsent_image_by_messenger_kid_parent": false
}

